# 05 — Weather Forecasting (30-day horizon)

**Phase 2, Step 4** — the final deliverable of the weather core.

**Input:**  `data/processed/weather_features.csv`
**Output:** `data/processed/weather_forecast.csv` (750 rows = 5 cities × 30 horizons × 5 targets)

---

## Strategy: direct multi-horizon + linear interpolation

A 30-day forecast can be built two ways:

| Approach | Pros | Cons |
|---|---|---|
| **Recursive** (predict t+1, feed back, repeat) | 1 model | errors compound |
| **Direct multi-horizon** (separate model per h) | no compounding | 30× compute |
| **Sparse direct + interpolation** (this notebook) | balance | small accuracy gap between anchor horizons |

We train 5 models per target at horizons `{1, 3, 7, 14, 30}` (25 models total,
~150 s end-to-end) and linearly interpolate predictions at intermediate
horizons. This keeps the horizon ladder honest — every prediction uses
exclusively observed features — while staying computationally cheap.

## Forecast anchor

The anchor is the last observed day in `weather_features.csv` (2026-04-18 in
the current snapshot). Every forecast date = anchor + h days.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

from src.weather import forecast as fc
from src.utils.config import MODELS_DIR, PROCESSED_DIR, INTERIM_DIR
from src.utils.logging_utils import get_logger
logger = get_logger("nb.05_forecasting")

## 2. Train the sparse horizon ladder

One model per (target, horizon) for horizons {1, 3, 7, 14, 30}.
Each is persisted to `models/weather/<target>/h{H}/<algo>.joblib`.

Run this once; subsequent notebook runs can skip straight to section 3.

In [ ]:
# Uncomment to retrain the ladder (~2-3 min)
# summary = fc.train_multi_horizon_models(
#     horizons=(1, 3, 7, 14, 30),
#     algo="hgbr",
# )

# Or read the already-saved summary
summary_path = MODELS_DIR / "weather" / "multi_horizon_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    print(f"Loaded cached summary from {summary_path}")
else:
    summary = fc.train_multi_horizon_models(horizons=(1,3,7,14,30), algo="hgbr")

summary[["target","horizon","MAE","RMSE","R2"]].round(3)

### 2a. Accuracy decay — how long does each target stay useful?

Higher bars = more degradation from the next-day skill. Temperature and
precipitation lose the most information over 30 days; wind direction was
already near its ceiling at h=1.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for target in summary["target"].unique():
    sub = summary[summary["target"]==target].sort_values("horizon")
    base = sub[sub["horizon"]==1]["RMSE"].iloc[0]
    ax.plot(sub["horizon"], sub["RMSE"] / base, "o-", lw=1.8, label=target)
ax.set_xlabel("horizon (days)")
ax.set_ylabel("RMSE / RMSE(h=1)")
ax.set_title("Forecast skill decay relative to h=1")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Emit the 30-day forecast dataset

`make_30day_forecast` loads the ladder, predicts at every horizon 1..30 per
city per target (interpolating between trained anchor horizons), and writes
`data/processed/weather_forecast.csv`.

In [ ]:
forecast = fc.make_30day_forecast(algo="hgbr")
print(f"Shape: {forecast.shape}")
print(f"Anchor date: {forecast['anchor_date'].iloc[0].date()}")
print(f"Forecast window: {forecast['forecast_date'].min().date()} -> {forecast['forecast_date'].max().date()}")
forecast.head()

### 3a. Tidy long -> wide pivot (convenience for joins)

In [ ]:
wide = fc.forecast_to_wide(forecast)
print(f"Wide shape: {wide.shape} (one row per City x forecast_date)")
wide.head(10)

## 4. Forecast QC — history + forecast overlays

If the forecast extends the history smoothly (no jump at the anchor date,
no physical impossibilities), the pipeline is working.

In [ ]:
hist = pd.read_csv(INTERIM_DIR / "weather_daily_clean.csv", parse_dates=["date"])
if hist["date"].dt.tz is not None:
    hist["date"] = hist["date"].dt.tz_localize(None)

anchor = forecast["anchor_date"].iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

def _plot_overlay(ax, target, y_col, ylabel, title, show_hist_from="2026-02-01"):
    for city in sorted(forecast["City"].unique()):
        h = hist[(hist["City"]==city) & (hist["date"] >= show_hist_from)]
        ax.plot(h["date"], h[y_col], lw=1.0, alpha=0.5)
        f = forecast[(forecast["City"]==city) & (forecast["target"]==target)]
        ax.plot(f["forecast_date"], f["y_pred"], lw=1.8, alpha=0.9, label=city)
    ax.axvline(anchor, c="k", ls=":", alpha=0.6)
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.legend(fontsize=7, ncol=5); ax.grid(alpha=0.3)

_plot_overlay(axes[0,0], "temperature_2m",      "temperature_2m_mean",
              "°C",  "Temperature — observed (thin) + forecast (bold)")
_plot_overlay(axes[0,1], "wind_speed_10m",      "wind_speed_10m_mean",
              "m/s", "Wind speed — observed + forecast")
_plot_overlay(axes[1,0], "rain",                "rain_sum",
              "mm",  "Rain — observed + forecast")
_plot_overlay(axes[1,1], "precipitation",       "precipitation_sum",
              "mm",  "Precipitation — observed + forecast")

plt.suptitle(f"30-day forecast anchored at {anchor.date()}", fontsize=12)
plt.tight_layout(); plt.show()

## 5. Per-city 30-day forecast summary

In [ ]:
wide_first_7 = (
    fc.forecast_to_wide(forecast)
      .sort_values(["City", "forecast_date"])
      .query("horizon_days <= 7")
      .set_index(["City", "forecast_date"])[[
          "temperature_2m", "wind_speed_10m", "wind_direction_10m", "rain", "precipitation"
      ]]
      .round(2)
)
wide_first_7

In [ ]:
# 30-day aggregated outlook per city
thirty_day = (
    forecast
    .groupby(["City", "target"])["y_pred"]
    .agg(["mean", "min", "max"])
    .round(2)
    .unstack("target")
)
thirty_day

## 6. Calibration summary

How far off should we expect each target to be? The test-set MAEs from the
ladder tell us.

In [ ]:
calibration = (
    summary
    .pivot_table(index="target", columns="horizon", values="MAE")
    .round(2)
)
calibration.columns = [f"h={h}" for h in calibration.columns]
calibration

## 7. Persisted artefacts

In [ ]:
out_files = [
    PROCESSED_DIR / "weather_forecast.csv",
    MODELS_DIR / "weather" / "multi_horizon_summary.csv",
]
for p in out_files:
    if p.exists():
        print(f"  {p.name:35s} {p.stat().st_size/1024:8.2f} KB")

# Model inventory
print("\nTrained models:")
for p in sorted((MODELS_DIR / "weather").rglob("*.joblib")):
    rel = p.relative_to(MODELS_DIR / "weather")
    print(f"  {str(rel):50s} {p.stat().st_size/1024:8.2f} KB")

---

## Hand-off to Phase 3 (Climate) and Phase 4 (Wildfire)

`data/processed/weather_forecast.csv` is now the canonical 30-day forecast:

- **Phase 3 (Climate)** will compare this forecast against historical same-month
  climatology to detect whether the next 30 days are anomalous.
- **Phase 4 (Wildfire)** will consume the `temperature_2m`, `wind_speed_10m`,
  `rain`, and `precipitation` predictions as predicted-weather inputs to the
  wildfire risk model.

**Phase 2 (Weather forecasting) ✅ complete.**